# PyTorch Data Loading for MAGIC Telescope Data

### Setup:

1. Need the `magic-gammas-chunked.parquet` and `magic-protons-chunked.parquet` files.
2. Need a python environment with python 3.10 or higher, pandas, numpy, torch, and matplotlib.
3. Optionally install ipython and ipykernel to use this notebook interactively.

In [ ]:
import random
from pathlib import Path
from typing import Literal

import pandas as pd
import pyarrow.parquet as pq
import torch
from torch.utils.data import ConcatDataset, DataLoader, Dataset, IterableDataset

### Load the data files
- Load parquet files into pandas DataFrames
- Might take minute or two to load into memory

In [ ]:
gamma_file = Path("/path/to/magic-gammas-chunked.parquet")
proton_file = Path("/path/to/magic-protons-chunked.parquet")

if not gamma_file.exists():
    raise FileNotFoundError(f"File {gamma_file} does not exist")
if not proton_file.exists():
    raise FileNotFoundError(f"File {proton_file} does not exist")

gammas = pd.read_parquet(gamma_file)
protons = pd.read_parquet(proton_file)

print(f"Loaded {len(gammas)} gamma events")
print(f"Loaded {len(protons)} proton events")

Loaded 374069 gamma events
Loaded 105174 proton events


## Create PyTorch Dataset class [loading entire dataset into memory]

**Note: this dataset class loads the entire dataset into memory. If you are working in a low-memory enviromnet, try using the iterable dataset below to stream in data from the disk**

- Custom Dataset class that efficiently loads events from pandas DataFrame
- Supports different input types (calibrated images, cleaned images, timing)
- Can combine images and timing as separate channels in the same tensor
- Can include additional image-level features (e.g., Hillas parameters) as a separate feature vector
- Automatically converts numpy arrays to torch tensors
- Always returns a dict with image, particle_id, event_number, run_number, and truth parameters

**Notes on data loading**
- `num_workers=0`: Required for notebooks because classes defined in notebook cells can't be pickled by multiprocessing workers. To enable multiprocessing:
  1. Move `MAGICDataset` class to a separate Python file (e.g., `magic_dataset.py`)
  2. Import it: `from magic_dataset import MAGICDataset`
  3. Then you can set `num_workers > 0` for faster data loading
- `pin_memory=True`: Faster GPU transfer (only use when GPU is available)
- `persistent_workers=True`: Keeps workers alive between epochs (only works when `num_workers > 0`)

In [18]:
class MAGICDataset(Dataset):
    """PyTorch Dataset for MAGIC telescope data.

    Args:
        dataframe: pandas DataFrame containing event data
        particle_type: "gamma" or "hadron" (for labeling)
        telescope_id: Which telescope to use (1 or 2, or "M1"/"M2")
        image_type: "calibrated", "clean", or "timing" - which image data to return
        use_both_telescopes: If True, returns both M1 and M2 images as separate channels
        include_timing: If True, stacks timing data as a separate channel after image data
        additional_features: List of column names to include as additional feature vector
            (e.g., ["hillas_width_m1", "hillas_length_m1", "hillas_size_m1"])
    """

    def __init__(
        self,
        dataframe: pd.DataFrame,
        particle_type: Literal["gamma", "hadron"],
        telescope_id: Literal[1, 2, "M1", "M2"] = 1,
        image_type: Literal["calibrated", "clean", "timing"] = "clean",
        use_both_telescopes: bool = False,
        include_timing: bool = False,
        additional_features: list[str] | None = None,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.particle_type = particle_type
        self.telescope_id = telescope_id
        self.image_type = image_type
        self.use_both_telescopes = use_both_telescopes
        self.include_timing = include_timing
        self.additional_features = additional_features if additional_features else []

        # Convert telescope_id to consistent format
        if telescope_id == 1 or telescope_id == "M1":
            self.telescope_key = "m1"
        else:
            self.telescope_key = "m2"

        # Label: 0 for gamma, 1 for hadron
        self.particle_id = 0 if particle_type == "gamma" else 1
        
        # Validate that include_timing is not used with image_type="timing"
        if include_timing and image_type == "timing":
            raise ValueError("Cannot use include_timing=True with image_type='timing'")
        
        # Validate that additional_features columns exist in dataframe
        if self.additional_features:
            missing_cols = [col for col in self.additional_features if col not in self.dataframe.columns]
            if missing_cols:
                raise ValueError(f"Columns not found in dataframe: {missing_cols}")

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, idx: int) -> dict:
        """Get a single event.

        Returns:
            dict with the following keys:
            - 'image': torch.Tensor with shape [channels, pixels]
            - 'particle_id': torch.Tensor (0 for gamma, 1 for hadron)
            - 'event_number': int
            - 'run_number': int
            - 'truth': dict containing 'particle_id', 'energy', 'theta', 'phi', 'first_interaction_height'
            - 'features': torch.Tensor (only if additional_features is provided)
        """
        row = self.dataframe.iloc[idx]

        # Get image data based on image_type
        if self.image_type == "calibrated":
            if self.use_both_telescopes:
                image_m1 = torch.from_numpy(row["image_m1"]).float()
                image_m2 = torch.from_numpy(row["image_m2"]).float()
                image = torch.stack([image_m1, image_m2], dim=0)  # Shape: [2, 1039]
            else:
                image_key = f"image_{self.telescope_key}"
                image = torch.from_numpy(row[image_key]).float()
        elif self.image_type == "clean":
            if self.use_both_telescopes:
                image_m1 = torch.from_numpy(row["clean_image_m1"]).float()
                image_m2 = torch.from_numpy(row["clean_image_m2"]).float()
                image = torch.stack([image_m1, image_m2], dim=0)  # Shape: [2, 1039]
            else:
                image_key = f"clean_image_{self.telescope_key}"
                image = torch.from_numpy(row[image_key]).float()
        elif self.image_type == "timing":
            if self.use_both_telescopes:
                timing_m1 = torch.from_numpy(row["timing_m1"]).float()
                timing_m2 = torch.from_numpy(row["timing_m2"]).float()
                image = torch.stack([timing_m1, timing_m2], dim=0)  # Shape: [2, 1039]
            else:
                timing_key = f"timing_{self.telescope_key}"
                image = torch.from_numpy(row[timing_key]).float()
        else:
            raise ValueError(f"Unknown image_type: {self.image_type}")

        # If include_timing is True, stack timing data as additional channel(s)
        if self.include_timing:
            if self.use_both_telescopes:
                timing_m1 = torch.from_numpy(row["timing_m1"]).float()
                timing_m2 = torch.from_numpy(row["timing_m2"]).float()
                timing = torch.stack([timing_m1, timing_m2], dim=0)  # Shape: [2, 1039]
                # Stack images first, then timing: [image_m1, image_m2, timing_m1, timing_m2]
                image = torch.cat([image, timing], dim=0)  # Shape: [4, 1039]
            else:
                timing_key = f"timing_{self.telescope_key}"
                timing = torch.from_numpy(row[timing_key]).float()
                # Stack image and timing: [image, timing]
                image = torch.stack([image, timing], dim=0)  # Shape: [2, 1039]
        else:
            # Ensure image is 2D (add channel dimension if single telescope and not including timing)
            if not self.use_both_telescopes:
                image = image.unsqueeze(0)  # Shape: [1, 1039]

        # Extract additional features if requested
        features = None
        if self.additional_features:
            feature_values = [row[col] for col in self.additional_features]
            features = torch.tensor(feature_values, dtype=torch.float32)

        result = {
            "image": image,
            "particle_id": torch.tensor(self.particle_id, dtype=torch.long),
            "event_number": row["event_number"],
            "run_number": row["run_number"],
            "truth": {
                "particle_id": torch.tensor(self.particle_id, dtype=torch.int32),
                "energy": torch.tensor(row["true_energy"], dtype=torch.float32),
                "theta": torch.tensor(row["true_theta"], dtype=torch.float32),
                "phi": torch.tensor(row["true_phi"], dtype=torch.float32),
                "first_interaction_height": torch.tensor(
                    row["true_first_interaction_height"], dtype=torch.float32
                ),
            },
        }
        if features is not None:
            result["features"] = features

        return result


### Create datasets and dataloaders
- Combine gamma and proton datasets
- Create train/validation/test splits
- Set up DataLoaders with proper batching

In [20]:
# Create datasets for gamma and proton events
gamma_dataset = MAGICDataset(gammas, particle_type="gamma", image_type="clean")
proton_dataset = MAGICDataset(protons, particle_type="hadron", image_type="clean")

# If you are doing a training with protons and gammas together: combine datasets using ConcatDataset

combined_dataset = ConcatDataset([gamma_dataset, proton_dataset])

print(f"Total events: {len(combined_dataset)}")
print(f"  - Gamma events: {len(gamma_dataset)}")
print(f"  - Proton events: {len(proton_dataset)}")

Total events: 297968
  - Gamma events: 192794
  - Proton events: 105174


In [7]:
# Create train/validation/test splits
# Using 70/15/15 split
total_size = len(combined_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    combined_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42),  # For reproducibility
)

print(f"Train set: {len(train_dataset)} events")
print(f"Validation set: {len(val_dataset)} events")
print(f"Test set: {len(test_dataset)} events")

Train set: 208577 events
Validation set: 44695 events
Test set: 44696 events


In [12]:
# Create DataLoaders
batch_size = 4
# Note: num_workers=0 for notebooks because classes defined in notebook cells
# can't be pickled by multiprocessing workers. Set to >0 only if you move
# MAGICDataset to a separate .py file and import it.
num_workers = 0  # Set to 0 for notebooks, or move Dataset class to separate file

# Only use pin_memory if you have a GPU
use_gpu = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=use_gpu,  # Faster GPU transfer (only if GPU available)
    persistent_workers=False,  # Not needed when num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=use_gpu,
    persistent_workers=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=use_gpu,
    persistent_workers=False,
)

print("DataLoaders created successfully!")
if num_workers == 0:
    print("  - Using single-threaded data loading (num_workers=0)")
    print("  - To enable multiprocessing, move MAGICDataset to a separate .py file")

DataLoaders created successfully!
  - Using single-threaded data loading (num_workers=0)
  - To enable multiprocessing, move MAGICDataset to a separate .py file


## Create PyTorch Dataset class [data streaming]

**Note: this dataset class streams the dataset from the file on the disk, saving on RAM**

**You need to use the `*-chunked.parquet` files for this to work**

Notes:
- Custom Dataset class that efficiently streams events from parquet files on disk
- Supports different input types (calibrated images, cleaned images, timing)
- Can combine images and timing as separate channels in the same tensor
- Can include additional image-level features (e.g., Hillas parameters) as a separate feature vector
- Automatically converts numpy arrays to torch tensors
- Always returns a dict with image, particle_id, event_number, run_number, and truth parameters
- Loads data in small groups (about 5k events from each file at a time, so ~tens of MB).

**Notes on data loading**
- `num_workers=0`: Required for notebooks because classes defined in notebook cells can't be pickled by multiprocessing workers. To enable multiprocessing:
  1. Move `MAGICStreamingDataset` class (with the the entire following cell) to a separate Python file (e.g., `magic_dataset.py`)
  2. Import it: `from magic_dataset import MAGICStreamingDataset`
  3. Then you can set `num_workers > 0` for faster data loading
- `pin_memory=True`: Faster GPU transfer (only use when GPU is available)
- `persistent_workers=True`: Keeps workers alive between epochs (only works when `num_workers > 0`)

In [120]:
class MAGICStreamingDataset(IterableDataset):
    """Streams MAGIC data from parquet without loading full file into RAM.

    Only one row group is held in memory at a time (about 5k events at a time, so ~tens of MB).
    Supports row-group-level + intra-group shuffling for training.

    Args:
        parquet_path: Path to the parquet file.
        particle_type: "gamma" or "hadron" (for labeling).
        telescope_id: Which telescope to use (1 or 2, or "M1"/"M2").
        image_type: "calibrated", "clean", or "timing".
        use_both_telescopes: If True, returns both M1 and M2 as
            separate channels.
        include_timing: If True, stacks timing data as extra channel(s).
        additional_features: Column names to include as a feature vector.
        row_groups: List of row group indices to iterate over.
            Use this for train/val/test splits. None = all groups.
        shuffle: If True, shuffles row group order and rows within
            each group.
        seed: Random seed for reproducibility.
        max_events: Maximum number of events to load. None = all events.
    """

    def __init__(
        self,
        parquet_path: str | Path,
        particle_type: Literal["gamma", "hadron"],
        telescope_id: Literal[1, 2, "M1", "M2"] = 1,
        image_type: Literal["calibrated", "clean", "timing"] = "clean",
        use_both_telescopes: bool = False,
        include_timing: bool = False,
        additional_features: list[str] | None = None,
        row_groups: list[int] | None = None,
        shuffle: bool = True,
        seed: int = 42,
        max_events: int | None = None,
    ):
        self.parquet_path = Path(parquet_path)
        self.particle_type = particle_type
        self.image_type = image_type
        self.use_both_telescopes = use_both_telescopes
        self.include_timing = include_timing
        self.additional_features = additional_features or []
        self.shuffle = shuffle
        self.seed = seed

        self.telescope_key = (
            "m1" if telescope_id in (1, "M1") else "m2"
        )
        self.particle_id = 0 if particle_type == "gamma" else 1

        if include_timing and image_type == "timing":
            raise ValueError(
                "Cannot use include_timing=True with image_type='timing'"
            )

        # Read metadata only — no data loaded
        self._pf = pq.ParquetFile(self.parquet_path)
        self.num_row_groups = self._pf.metadata.num_row_groups
        self._row_group_sizes = [
            self._pf.metadata.row_group(i).num_rows
            for i in range(self.num_row_groups)
        ]

        # Filter to request specific row groups (needed for train/val/test splits)
        if row_groups is not None:
            self._active_groups = row_groups
        else:
            self._active_groups = list(range(self.num_row_groups))

        self._total_rows = sum(
            self._row_group_sizes[i] for i in self._active_groups
        )
        
        # set a cap on the total rows if requested
        self.max_events = max_events
        if self.max_events is not None:
            self._total_rows = min(self._total_rows, self.max_events)

        # Only read columns we actually need
        self._columns = self._required_columns()

    def _required_columns(self) -> list[str]:
        cols = {
            "event_number",
            "run_number",
            "true_energy",
            "true_theta",
            "true_phi",
            "true_first_interaction_height",
        }
        tkeys = (
            ["m1", "m2"]
            if self.use_both_telescopes
            else [self.telescope_key]
        )
        for tk in tkeys:
            if self.image_type == "calibrated":
                cols.add(f"image_{tk}")
            elif self.image_type == "clean":
                cols.add(f"clean_image_{tk}")
            elif self.image_type == "timing":
                cols.add(f"timing_{tk}")
            if self.include_timing:
                cols.add(f"timing_{tk}")
        cols.update(self.additional_features)
        return list(cols)

    def __len__(self) -> int:
        return self._total_rows

    def _process_row(self, row) -> dict:
        tk = self.telescope_key
        both = self.use_both_telescopes

        if self.image_type == "calibrated":
            prefix = "image"
        elif self.image_type == "clean":
            prefix = "clean_image"
        else:
            prefix = "timing"

        if both:
            im1 = torch.from_numpy(row[f"{prefix}_m1"]).float()
            im2 = torch.from_numpy(row[f"{prefix}_m2"]).float()
            image = torch.stack([im1, im2], dim=0)
        else:
            image = torch.from_numpy(row[f"{prefix}_{tk}"]).float()

        if self.include_timing:
            if both:
                t1 = torch.from_numpy(row["timing_m1"]).float()
                t2 = torch.from_numpy(row["timing_m2"]).float()
                timing = torch.stack([t1, t2], dim=0)
                image = torch.cat([image, timing], dim=0)
            else:
                timing = torch.from_numpy(row[f"timing_{tk}"]).float()
                image = torch.stack([image, timing], dim=0)
        elif not both:
            image = image.unsqueeze(0)

        result = {
            "image": image,
            "particle_id": torch.tensor(
                self.particle_id, dtype=torch.long
            ),
            "event_number": row["event_number"],
            "run_number": row["run_number"],
            "truth": {
                "particle_id": torch.tensor(
                    self.particle_id, dtype=torch.int32
                ),
                "energy": torch.tensor(
                    row["true_energy"], dtype=torch.float32
                ),
                "theta": torch.tensor(
                    row["true_theta"], dtype=torch.float32
                ),
                "phi": torch.tensor(
                    row["true_phi"], dtype=torch.float32
                ),
                "first_interaction_height": torch.tensor(
                    row["true_first_interaction_height"],
                    dtype=torch.float32,
                ),
            },
        }
        if self.additional_features:
            result["features"] = torch.tensor(
                [row[c] for c in self.additional_features],
                dtype=torch.float32,
            )
        return result

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        group_indices = list(self._active_groups)

        # Worker-specific seed avoids identical shuffle streams across workers.
        if worker_info is None:
            worker_seed = self.seed
        else:
            worker_seed = (self.seed + worker_info.seed) % (2**32)

        rng = random.Random(worker_seed)

        if worker_info is not None:
            per_worker = len(group_indices) // worker_info.num_workers
            start = worker_info.id * per_worker
            end = (
                start + per_worker
                if worker_info.id < worker_info.num_workers - 1
                else len(group_indices)
            )
            group_indices = group_indices[start:end]

        if self.shuffle:
            rng.shuffle(group_indices)

        emitted = 0
        for gi in group_indices:
            if self.max_events is not None and emitted >= self.max_events:
                break

            table = self._pf.read_row_group(gi, columns=self._columns)
            df = table.to_pandas()

            if self.shuffle:
                row_seed = rng.randint(0, 2**32 - 1)
                df = df.sample(frac=1, random_state=row_seed)

            for _, row in df.iterrows():
                if self.max_events is not None and emitted >= self.max_events:
                    break
                yield self._process_row(row)
                emitted += 1

            del df, table
            
            
class StreamingDatasetShuffler(IterableDataset):
    """This makes sure that gamma and proton events are fully mixed together (interleaved)
    rather than appearing in sequence.
    
    NOTE: you only need this if you are training a classifier. If you only work with gamma-ray data, for example, 
    you can skip this.

    Args:
        dataset: The IterableDataset to shuffle.
        buffer_size: Number of samples to hold in the buffer.
            Larger = better randomness, more RAM. 10k samples is
            typically only a few MB of overhead.
    """

    def __init__(self, dataset: IterableDataset, buffer_size: int = 10_000):
        self.dataset = dataset
        self.buffer_size = buffer_size

    def __len__(self) -> int:
        return len(self.dataset)

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        seed = 42 if worker_info is None else worker_info.seed % (2**32)
        rng = random.Random(seed)

        buf = []
        for sample in self.dataset:
            buf.append(sample)
            if len(buf) >= self.buffer_size:
                idx = rng.randint(0, len(buf) - 1)
                yield buf.pop(idx)
        # Flush remaining
        rng.shuffle(buf)
        yield from buf
        
        
class InterleavedDataset(IterableDataset):
    """Interleaves samples from multiple IterableDatasets by randomly
    drawing from each, so classes are mixed from the start."""

    def __init__(self, datasets: list[IterableDataset]):
        self.datasets = datasets

    def __len__(self) -> int:
        return sum(len(ds) for ds in self.datasets)

    def __iter__(self):
        iterators = [iter(ds) for ds in self.datasets]
        active = list(range(len(iterators)))

        worker_info = torch.utils.data.get_worker_info()
        seed = 42 if worker_info is None else worker_info.seed % (2**32)
        rng = random.Random(seed)

        while active:
            i = rng.choice(active)
            try:
                yield next(iterators[i])
            except StopIteration:
                active.remove(i)
        

def split_row_groups(
    parquet_path: str | Path,
    train_frac: float = 0.7,
    val_frac: float = 0.15,
    seed: int = 42,
    verbose: bool = False
) -> dict[str, list[int]]:
    """Shuffle and split row group indices into train/val/test.

    Args:
        parquet_path: Path to the parquet file.
        train_frac: Fraction of row groups for training.
        val_frac: Fraction of row groups for validation.
            Test gets the remainder (1 - train_frac - val_frac).
        seed: Random seed for reproducible splits.
        verbose: Whether to print verbose output.
    Returns:
        Dict with keys "train", "val", "test", each mapping
        to a list of row group indices.
    """
    pf = pq.ParquetFile(parquet_path)
    indices = list(range(pf.metadata.num_row_groups))

    rng = random.Random(seed)
    rng.shuffle(indices)

    n = len(indices)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)

    splits = {
        "train": sorted(indices[:n_train]),
        "val": sorted(indices[n_train : n_train + n_val]),
        "test": sorted(indices[n_train + n_val :]),
    }

    # Print summary
    for name, groups in splits.items():
        n_rows = sum(
            pf.metadata.row_group(i).num_rows for i in groups
        )
        if verbose:
            print(
                f"  {name}: {len(groups)} row groups, "
                f"{n_rows:,} events"
            )

    return splits

In [132]:
if not gamma_file.exists():
    raise FileNotFoundError(f"File {gamma_file} does not exist")
if not proton_file.exists():
    raise FileNotFoundError(f"File {proton_file} does not exist")

train_frac = 0.7  # 70% of data for training
val_frac = 0.15  # 15% of data for validation
test_frac = 1 - train_frac - val_frac  # 15% of data for testing

split_dict = {
    "train": train_frac,
    "val": val_frac,
    "test": test_frac
}

# Split each file independently (preserves gamma/proton class ratio)
gamma_splits = split_row_groups(gamma_file, seed=42, train_frac=train_frac, val_frac=val_frac)
proton_splits = split_row_groups(proton_file, seed=42, train_frac=train_frac, val_frac=val_frac)

for k, v in split_dict.items():
    print(f"Using {round(v*100, 2)}% of the data for {k} split")

Using 70.0% of the data for train split
Using 15.0% of the data for val split
Using 15.0% of the data for test split


- `batch_size`: how many events to train on simultaneously during training. will depend on your GPU memory, you might have to play around with this.
- `max_events`: how many events to train on in total from each dataset. We have just over 100k protons and 300k gammas, so going over 100k is not a great idea for classification-- you might make a class inbalance!
- `num_workers`: how many CPU cores to use for data loading. Set to 0 if you are working in a notebook.
- `shuffle`: whether to shuffle the data. always recommended to set to True.
- `use_gpu`: whether to use a GPU. If you have one, set to True. pytorch can detect if you have one and set this automatically.

In [ ]:

batch_size = 8
max_events = 100_000  
num_workers = 0 
shuffle = True
use_gpu = torch.cuda.is_available()

_event_counter = 0
for split_name in ["train", "val", "test"]:
    
    gamma_ds = MAGICStreamingDataset(
        gamma_file,
        particle_type="gamma",
        image_type="clean",
        shuffle=(split_name == "train"),  # Only shuffle training data
        row_groups=gamma_splits[split_name],
        max_events=int(max_events * split_dict[split_name]),
    )
    
    # Following only necessary if you are training a classifier.
    proton_ds = MAGICStreamingDataset(
        proton_file,
        particle_type="hadron",
        image_type="clean",
        shuffle=(split_name == "train"),  # Only shuffle training data
        row_groups=proton_splits[split_name],
        max_events=int(max_events * split_dict[split_name]),
    )
    
    # NOTE: the following line is only needed if you are training a classifier. 
    # If you only work with gamma-ray data, for example, 
    # you can skip this.
    combined = InterleavedDataset([gamma_ds, proton_ds])
    
    print(f"{split_name} dataset length: {len(combined)} events")
    
    _event_counter += len(combined)
    
    # Wrap in shuffle buffer for gamma/proton interleaving
    if shuffle:
        combined = StreamingDatasetShuffler(combined, buffer_size=5_000)
    
    if split_name == "train":
        train_loader = DataLoader(
            combined,
            batch_size=batch_size,
            num_workers=num_workers,
            pin_memory=use_gpu,
        )
        
    elif split_name == "val":
        val_loader = DataLoader(
            combined,
            batch_size=batch_size,
            num_workers=num_workers,
            pin_memory=use_gpu,
        )
        
    elif split_name == "test":
        test_loader = DataLoader(
            combined,
            batch_size=batch_size,
            num_workers=num_workers,
            pin_memory=use_gpu,
        )
        
    else:
        raise ValueError(f"Invalid split name: {split_name}")
    
print(f"Total events: {_event_counter}")

train dataset length: 140000 events
val dataset length: 30000 events
test dataset length: 30000 events
Total events: 200000


In [125]:
# get some data from the first batch
batch = next(iter(train_loader))

print(f"Keys returned by the DataLoader: {list(batch.keys())}")
print(f"\nImage shape: {batch['image'].shape}")
print("  Expected: [batch_size, channels, pixels]")
print(f"\nGammas in batch: {(batch['particle_id'] == 0).sum().item()}")
print(f"Protons in batch: {(batch['particle_id'] == 1).sum().item()}")
print("\nBatch Image stats:")
print(f"  Min pixel value:  {batch['image'].min():.2f}")
print(f"  Max pixel value:  {batch['image'].max():.2f}")
print(f"  Mean pixel value: {batch['image'].mean():.2f}")

Keys returned by the DataLoader: ['image', 'particle_id', 'event_number', 'run_number', 'truth']

Image shape: torch.Size([8, 1, 1183])
  Expected: [batch_size, channels, pixels]

Gammas in batch: 6
Protons in batch: 2

Batch Image stats:
  Min pixel value:  0.00
  Max pixel value:  128.50
  Mean pixel value: 0.57


## Test the data loading
- Check batch shapes and types
- Verify particle_id labels are correct
- Note: DataLoader returns batches of dicts, so each key is batched automatically

In [131]:
# Get a batch from the training loader
# you can keep re-running this cell to see different batches
batch = next(iter(train_loader))

print("Keys returned by the DataLoader:", list(batch.keys()))
print(f"\nImage shape: {batch['image'].shape}")
print("  - Expected: [batch_size, channels, pixels]")
print(f"  - Actual: [{batch['image'].shape[0]}, {batch['image'].shape[1]}, {batch['image'].shape[2]}]")
print(f"\nParticle ID shape: {batch['particle_id'].shape}")
print(f"Particle ID dtype: {batch['particle_id'].dtype}")
print(f"\nSample particle IDs: {batch['particle_id'][:10]}")
print("  - 0 = gamma, 1 = hadron/proton")
print(f" - Gammas in batch: {(batch['particle_id'] == 0).sum().item()}")
print(f" - Protons in batch: {(batch['particle_id'] == 1).sum().item()}")
print("\nImage batch pixel values stats:")
print(f"  - Min: {batch['image'].min():.2f}")
print(f"  - Max: {batch['image'].max():.2f}")
print(f"  - Mean: {batch['image'].mean():.2f}")
print(f"  - Std: {batch['image'].std():.2f}")

Keys returned by the DataLoader: ['image', 'particle_id', 'event_number', 'run_number', 'truth']

Image shape: torch.Size([8, 1, 1183])
  - Expected: [batch_size, channels, pixels]
  - Actual: [8, 1, 1183]

Particle ID shape: torch.Size([8])
Particle ID dtype: torch.int64

Sample particle IDs: tensor([1, 1, 1, 0, 1, 0, 1, 0])
  - 0 = gamma, 1 = hadron/proton
 - Gammas in batch: 3
 - Protons in batch: 5

Image batch pixel values stats:
  - Min: 0.00
  - Max: 37.62
  - Mean: 0.34
  - Std: 2.23


## Examples

**Note: these  upcoming examples are using the standard (not streaming) dataset classes, but they can be adapted to the streaming case.**

### Example: Using both telescopes
- Load data from both M1 and M2 telescopes as separate channels

In [14]:
# Create dataset with both telescopes
gamma_dataset_both = MAGICDataset(
    gammas,
    particle_type="gamma",
    image_type="clean",
    use_both_telescopes=True,
)
proton_dataset_both = MAGICDataset(
    protons,
    particle_type="hadron",
    image_type="clean",
    use_both_telescopes=True,
)

combined_dataset_both = ConcatDataset([gamma_dataset_both, proton_dataset_both])
train_dataset_both, val_dataset_both, test_dataset_both = torch.utils.data.random_split(
    combined_dataset_both,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42),
)

train_loader_both = DataLoader(
    train_dataset_both,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,  # Use 0 for notebooks
    pin_memory=use_gpu,
)

# Check the shape with both telescopes
batch_both = next(iter(train_loader_both))
images_both = batch_both["image"]
particle_ids_both = batch_both["particle_id"]

print(f"Batch shape with both telescopes: {images_both.shape}")
print("  - Expected: [batch_size, 2, pixels]")
print(
    f"  - Actual: [{images_both.shape[0]}, {images_both.shape[1]}, {images_both.shape[2]}]"
)
print("  - Channel 0: M1 telescope")
print("  - Channel 1: M2 telescope")
print(f"\nParticle IDs: {particle_ids_both[:10]}")
print("  - 0 = gamma, 1 = hadron/proton")

Batch shape with both telescopes: torch.Size([4, 2, 1183])
  - Expected: [batch_size, 2, pixels]
  - Actual: [4, 2, 1183]
  - Channel 0: M1 telescope
  - Channel 1: M2 telescope

Particle IDs: tensor([0, 1, 0, 1])
  - 0 = gamma, 1 = hadron/proton


### Example: Using image + timing data
- Load cleaned images AND timing information as separate channels in the same tensor
- Channel 0: cleaned image (photoelectrons)
- Channel 1: timing data (nanoseconds)

In [15]:
# Create dataset with cleaned images AND timing data as separate channels
# Channel 0: cleaned image, Channel 1: timing data
gamma_dataset_timing = MAGICDataset(
    gammas,
    particle_type="gamma",
    image_type="clean",
    telescope_id=1,
    include_timing=True,  # Add timing as a separate channel
)
proton_dataset_timing = MAGICDataset(
    protons,
    particle_type="hadron",
    image_type="clean",
    telescope_id=1,
    include_timing=True,  # Add timing as a separate channel
)

combined_dataset_timing = ConcatDataset([gamma_dataset_timing, proton_dataset_timing])
train_dataset_timing, val_dataset_timing, test_dataset_timing = (
    torch.utils.data.random_split(
        combined_dataset_timing,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42),
    )
)

train_loader_timing = DataLoader(
    train_dataset_timing,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,  # Use 0 for notebooks
    pin_memory=use_gpu,
)

# Check the shape with image + timing data
batch_timing = next(iter(train_loader_timing))
images_timing = batch_timing["image"]
particle_ids_timing = batch_timing["particle_id"]

print(f"Batch shape with image + timing: {images_timing.shape}")
print("  - Expected: [batch_size, 2, pixels]")
print("  - Channel 0: cleaned image")
print("  - Channel 1: timing data")
print("\nImage channel (channel 0) stats:")
print(f"  - Min: {images_timing[:, 0, :].min():.2f} p.e.")
print(f"  - Max: {images_timing[:, 0, :].max():.2f} p.e.")
print(f"  - Mean: {images_timing[:, 0, :].mean():.2f} p.e.")
print("\nTiming channel (channel 1) stats:")
print(f"  - Min: {images_timing[:, 1, :].min():.2f} ns")
print(f"  - Max: {images_timing[:, 1, :].max():.2f} ns")
print(f"  - Mean: {images_timing[:, 1, :].mean():.2f} ns")
print(f"  - Std: {images_timing[:, 1, :].std():.2f} ns")
print(f"\nParticle IDs: {particle_ids_timing[:10]}")
print("  - 0 = gamma, 1 = hadron/proton")

Batch shape with image + timing: torch.Size([4, 2, 1183])
  - Expected: [batch_size, 2, pixels]
  - Channel 0: cleaned image
  - Channel 1: timing data

Image channel (channel 0) stats:
  - Min: 0.00 p.e.
  - Max: 187.50 p.e.
  - Mean: 0.85 p.e.

Timing channel (channel 1) stats:
  - Min: -163.50 ns
  - Max: 80.25 ns
  - Mean: 35.72 ns
  - Std: 16.00 ns

Particle IDs: tensor([0, 1, 0, 0])
  - 0 = gamma, 1 = hadron/proton


### Example: Using additional features (Hillas parameters)
- Include image-level parameters like Hillas parameters as a separate feature vector
- Useful for combining image data with reconstructed parameters

In [16]:
# Create dataset with cleaned images AND Hillas parameters as additional features
# The features will be returned as a separate tensor alongside the image
hillas_features = [
    "hillas_width_m1",
    "hillas_length_m1",
    "hillas_size_m1",
    "hillas_width_m2",
    "hillas_length_m2",
    "hillas_size_m2",
]

gamma_dataset_features = MAGICDataset(
    gammas,
    particle_type="gamma",
    image_type="clean",
    telescope_id=1,
    additional_features=hillas_features,
)
proton_dataset_features = MAGICDataset(
    protons,
    particle_type="hadron",
    image_type="clean",
    telescope_id=1,
    additional_features=hillas_features,
)

combined_dataset_features = ConcatDataset([gamma_dataset_features, proton_dataset_features])
train_dataset_features, val_dataset_features, test_dataset_features = (
    torch.utils.data.random_split(
        combined_dataset_features,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42),
    )
)

train_loader_features = DataLoader(
    train_dataset_features,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,  # Use 0 for notebooks
    pin_memory=use_gpu,
)

# Check the shape with image + features
batch_features = next(iter(train_loader_features))
images_features = batch_features["image"]
features = batch_features["features"]
particle_ids_features = batch_features["particle_id"]

print(f"Batch shape - images: {images_features.shape}")
print(f"Batch shape - features: {features.shape}")
print(f"Batch shape - particle_ids: {particle_ids_features.shape}")
print(f"\nFeatures included: {hillas_features}")
print("\nFeature stats (first batch):")
for i, feat_name in enumerate(hillas_features):
    print(f"  - {feat_name}: min={features[:, i].min():.2f}, max={features[:, i].max():.2f}, mean={features[:, i].mean():.2f}")
print(f"\nParticle IDs: {particle_ids_features[:10]}")
print("  - 0 = gamma, 1 = hadron/proton")

Batch shape - images: torch.Size([4, 1, 1183])
Batch shape - features: torch.Size([4, 6])
Batch shape - particle_ids: torch.Size([4])

Features included: ['hillas_width_m1', 'hillas_length_m1', 'hillas_size_m1', 'hillas_width_m2', 'hillas_length_m2', 'hillas_size_m2']

Feature stats (first batch):
  - hillas_width_m1: min=23.19, max=55.32, mean=33.13
  - hillas_length_m1: min=45.09, max=182.60, mean=83.70
  - hillas_size_m1: min=184.50, max=1756.82, mean=846.60
  - hillas_width_m2: min=7.42, max=59.20, mean=28.73
  - hillas_length_m2: min=13.98, max=126.16, mean=59.48
  - hillas_size_m2: min=31.05, max=3145.56, mean=974.34

Particle IDs: tensor([1, 1, 1, 0])
  - 0 = gamma, 1 = hadron/proton


### Example: Accessing metadata
- All datasets always return event numbers, run numbers, and truth values
- Truth parameters include particle_id, energy, theta, phi, and first_interaction_height

In [17]:
# Get a single event (all datasets return metadata by default)
sample_event = gamma_dataset[0]

print("Event metadata:")
print(f"  - Event number: {sample_event['event_number']}")
print(f"  - Run number: {sample_event['run_number']}")
print(
    f"  - Particle ID: {sample_event['particle_id'].item()} ({'gamma' if sample_event['particle_id'].item() == 0 else 'hadron'})"
)
print(f"  - Image shape: {sample_event['image'].shape}")
print("\nTruth values:")
for key, value in sample_event["truth"].items():
    if isinstance(value, torch.Tensor):
        print(f"  - {key}: {value.item():.4f}")
    else:
        print(f"  - {key}: {value}")

Event metadata:
  - Event number: 21
  - Run number: 821318
  - Particle ID: 0 (gamma)
  - Image shape: torch.Size([1, 1183])

Truth values:
  - particle_id: 0.0000
  - energy: 341.8620
  - theta: 0.5189
  - phi: 2.7471
  - first_interaction_height: 1279174.5000
